In [ ]:
# Modified by Mehrnoosh

# Ignore Warnings
import warnings
warnings.filterwarnings('ignore')
import openai
from openai import OpenAI
import pandas as pd
from typing import List
from datetime import datetime
from time import sleep
from os import path
import json

#Set root path to the location of open oai_finetune_modules
import os
os.chdir("/llm_coding/")
import Mehr_oai_finetune_modules as oaimd

In [2]:

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_column', None)
pd.set_option('display.max_rows', None)

In [ ]:

ID_STRS = {
    "tgt": ("You are a member of a population that is a high priority target for efforts related to reducing "
           "incidents of the Humman Immunodefiency Virus (For ex: Men who have sex with men).")
}

Q_PROMPTS = {
    "act1": ("How likely is it that you would share this message with your "
            "social network if you encountered it on social media?"),
    "act2": ("How likely do you think it is that this message was specifically "
            "designed for gay or bisexual men"),
    "act3": ("How strongly do you feel that this message is convincing?"),
    "act4": ("How strongly do you feel this message gives the audience a clear indication "
            "of what to do about HIV prevention or testing"),
    "act5": ("How strongly do you feel this message points to concrete resources that may "
            "help to implement HIV prevention or testing"),
    "eff1": ("How strongly do you agree that this message is truthful"),
    "eff2": ("How strongly do you agree that this message seems to have been designed for "
            "the benefit of people with high risk of HIV (For ex: Men who have sex with men)"),
    "eff3": ("How strongly do you agree that this message has the potential to change the "
            "behaviour of people with high risk of HIV (For ex: Men who have sex with men)"),
    "eff4": ("How strongly do you agree that this message has the potential to change "
            "the behaviour of people with high risk of HIV (For ex: Men who have sex with men)")
}

ANNOTATION_TYPE_PROMPTS = {
    "rec_act": ("You will rate these tweets with integer values ranging from 1 to 4, where "
               "1 is \"Not at all\" and 4 is \"Definitely\", based on the following question."),
    "avg_act": ("You will rate these tweets from 1 to 4, where "
               "1 is \"Not at all\" and 4 is \"Definitely\", based on the following question."),
    "rec_eff": ("You will rate these tweets with integer values ranging from 1 to 3, where "
               "1 is \"Disagree\", 2 is \"Neither agree nor disagree\", and 3 is \"Agree\", "
               "based on the following question."),
    "avg_eff": ("You will rate these tweets from 1 to 3, where "
               "1 is \"Disagree\", 2 is \"Neither agree nor disagree\", and 3 is \"Agree\", "
               "based on the following question.")
}

# Initialize the OpenAI client
client = OpenAI(api_key= '' # Enter your openAI platform API key here)  

def annotate_tweet(tweet, role_id, prompt, annotation_type):
    """
    Annotate a single tweet using OpenAI's API
    
    :param tweet: Content of the tweet
    :param role_id: Role identifier from ID_STRS
    :param prompt: Specific prompt from Q_PROMPTS
    :param annotation_type: Annotation type from ANNOTATION_TYPE_PROMPTS
    :return: Annotation result
    """
    full_prompt = f"""{ID_STRS[role_id]}
{ANNOTATION_TYPE_PROMPTS[annotation_type]} 
{Q_PROMPTS[prompt]}
Tweet: {tweet}
Please provide your rating."""
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini-2024-07-18",
            messages=[
                {"role": "system", "content": ID_STRS[role_id]},
                {"role": "user", "content": full_prompt}
            ],
            max_tokens=20
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error annotating tweet: {e}")
        return None


def annotate_dataframe(df):
    """
    Annotate all tweets in the DataFrame with appropriate prompt-annotation type combinations
    
    :param df: DataFrame containing tweets in 'Content' column
    :return: DataFrame with annotations
    """
    annotation_columns = []
    
    # Generate appropriate column combinations
    for role_id in ID_STRS.keys():
        for prompt, annotation_types in get_prompt_annotation_pairs():
            for annotation_type in annotation_types:
                col_name = f"{role_id}_{prompt}_{annotation_type}"
                annotation_columns.append(col_name)
                df[col_name] = None
    
    # Annotate each tweet
    for index, row in df.iterrows():
        tweet = row['Content']
        for role_id in ID_STRS.keys():
            for prompt, annotation_types in get_prompt_annotation_pairs():
                for annotation_type in annotation_types:
                    col_name = f"{role_id}_{prompt}_{annotation_type}"
                    df.at[index, col_name] = annotate_tweet(tweet, role_id, prompt, annotation_type)
    
    return df

def get_prompt_annotation_pairs():
    """
    Returns appropriate pairs of prompts and their corresponding annotation types
    """
    pairs = []
    
    # For act prompts (act1 through act5)
    act_annotation_types = [at for at in ANNOTATION_TYPE_PROMPTS.keys() if at.endswith('act')]
    act_prompts = [p for p in Q_PROMPTS.keys() if p.startswith('act')]
    for prompt in act_prompts:
        pairs.append((prompt, act_annotation_types))
    
    # For eff prompts (eff1 through eff4)
    eff_annotation_types = [at for at in ANNOTATION_TYPE_PROMPTS.keys() if at.endswith('eff')]
    eff_prompts = [p for p in Q_PROMPTS.keys() if p.startswith('eff')]
    for prompt in eff_prompts:
        pairs.append((prompt, eff_annotation_types))
    
    return pairs

In [4]:
def main():
    start_time = time.time()
    
    # Load your DataFrame
    act_data = pd.read_csv('act_data.csv')  # Replace with your actual file
    # dff = act_data.head(5)
    
    # Annotate the DataFrame
    result_df = annotate_dataframe(act_data)
    
    # Save annotated results
    result_df.to_csv('annotated_tweets.csv', index=False)
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Program completed in {elapsed_time:.2f} seconds.")
    print("Annotation process completed.")

if __name__ == "__main__":
    main()

print("Tweet annotation script is ready. Make sure to set your OpenAI API key and prepare your input data.")
# Record the end time
end_time = time.time()

Program completed in 606.24 seconds.
Annotation process completed.
Tweet annotation script is ready. Make sure to set your OpenAI API key and prepare your input data.


In [5]:
data = pd.read_csv("annotated_tweets.csv")
data.head(5)

,sub_id,Content,label,info_sti_tst_num,info_sti_prev_num,info_sti_care_num,inst_sti_tst_num,inst_sti_prev_num,inst_sti_falseprev_num,inst_sti_care_num,negt_sti_tst_num,negt_pub_hlt_num,if_sti_num,whic_sti_num,info_any_num,inst3_any_num,inst4_any_num,negt_any_num,random_hiv_num,info_sti_tst_avg,info_sti_prev_avg,info_sti_care_avg,inst_sti_tst_avg,inst_sti_prev_avg,inst_sti_falseprev_avg,inst_sti_care_avg,negt_sti_tst_avg,negt_pub_hlt_avg,if_sti_avg,whic_sti_avg,info_any_avg,inst3_any_avg,inst4_any_avg,negt_any_avg,random_hiv_avg,neg_actionable,false_neg,tgt_act1_rec_act,tgt_act1_avg_act,tgt_act2_rec_act,tgt_act2_avg_act,tgt_act3_rec_act,tgt_act3_avg_act,tgt_act4_rec_act,tgt_act4_avg_act,tgt_act5_rec_act,tgt_act5_avg_act,tgt_eff1_rec_eff,tgt_eff1_avg_eff,tgt_eff2_rec_eff,tgt_eff2_avg_eff,tgt_eff3_rec_eff,tgt_eff3_avg_eff,tgt_eff4_rec_eff,tgt_eff4_avg_eff
0,actionable vetted_48,Use a condom and PrEP to protect yourself from HIV. With the condom you can also prevent unwanted pregnancies and STIs. So double up and stay safe! https://t.co/RWbyRfx3OL,actionable vetted,0.000000,1.0,0.000000,0.000000,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.000000,1.0,1.0,0.0,0.0,No,Yes,No,No,Yes,No,No,No,No,Yes,Yes,Yes,Yes,Yes,No,No,No,No,"I would rate this tweet a 4, ""Definitely."" The message promotes important health practices related to","I would rate this tweet a 4, ""Definitely"". The message is informative and emphasizes the importance","I would rate this tweet a 3. While it discusses the use of condoms and PrEP,",I would rate this tweet a 3. While the message encourages safe practices that are relevant to all,"I would rate this tweet a 4. The message is clear, straightforward, and emphasizes the importance","I would rate this tweet a 4, ""Definitely."" The message is clear and highlights the importance","I would rate this tweet a 4 (""Definitely""). The message clearly provides specific actions (using a","I would rate this tweet a 4: ""Definitely."" \n\nThe message is clear and provides specific",I would rate this tweet a 4. The message clearly points out concrete resources for HIV prevention (,"I would rate this tweet a 4, ""Definitely."" The message clearly points to concrete resources for","I would rate the tweet a 3: ""Agree."" The message is truthful as it accurately promotes","I would rate this tweet a 3, ""Agree."" The message is truthful as using condoms and",I would rate the tweet as a 3 (Agree). The message clearly emphasizes the importance of using,"I would rate this tweet a 3, ""Agree."" The message clearly emphasizes the importance of using","I would rate this tweet as a 3 - ""Agree."" The message clearly promotes the use of",3 - Agree,I would rate this tweet a 3 (Agree). The message promotes the use of condoms and Pr,"I would rate this tweet a 3 (""Agree""). The message clearly emphasizes the importance of using condoms"
1,actionable vetted_49,@laytee_02 Coz i do test for HIV and i always wear a condom,actionable vetted,1.000000,0.0,0.000000,0.666667,1.0,0.0,0.0,0.0,0.0,1.0,1.0,1.000000,1.0,1.0,0.0,0.0,Yes,No,No,Yes,Yes,No,No,No,No,Yes,Yes,Yes,Yes,Yes,No,No,No,No,"I would rate the tweet a 4. The message promotes safe practices and encourages HIV testing, which",I would rate this tweet a 3. It's a positive message about testing for HIV and practicing safe,I would rate the tweet a 2. While the mention of testing for HIV and wearing condoms is,I would rate this tweet a 2. While it mentions testing for HIV and wearing condoms—important,I would rate this tweet a 3. The message is quite convincing as it emphasizes the importance of,I would rate this tweet a 3. It conveys a positive message about taking proactive steps to prevent,"I would rate this tweet a 3. While it mentions testing for HIV and using condoms, which",I would rate this tweet a 3. It clearly indicates the actions of testing for HIV and wearing,"Rating: 2\n\nThe tweet mentions testing for HIV and wearing condoms, which are important aspects of",I would rate this tweet a 2. While it

In [6]:
# Annotating the negative tweets


def main():
    start_time = time.time()
    
    # Load your DataFrame
    neg_data = pd.read_csv('hiv_msgs_working_20241118.csv')  # Replace with your actual file
    neg_data = neg_data[neg_data['neg_revisit']=='Yes']
    # Annotate the DataFrame
    result_df = annotate_dataframe(neg_data)
    
    # Save annotated results
    result_df.to_csv('neg_annotated_tweets.csv', index=False)
    
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Program completed in {elapsed_time:.2f} seconds.")
    print("Annotation process completed.")

if __name__ == "__main__":
    main()

print("Tweet annotation script is ready. Make sure to set your OpenAI API key and prepare your input data.")
# Record the end time
end_time = time.time()

Program completed in 454.65 seconds.
Annotation process completed.
Tweet annotation script is ready. Make sure to set your OpenAI API key and prepare your input data.


In [7]:
data = pd.read_csv("neg_annotated_tweets.csv")
data.head(5)

,Unnamed: 0,id_str,Content,comments,true_neg?,neg_revisit,tgt_act1_rec_act,tgt_act1_avg_act,tgt_act2_rec_act,tgt_act2_avg_act,tgt_act3_rec_act,tgt_act3_avg_act,tgt_act4_rec_act,tgt_act4_avg_act,tgt_act5_rec_act,tgt_act5_avg_act,tgt_eff1_rec_eff,tgt_eff1_avg_eff,tgt_eff2_rec_eff,tgt_eff2_avg_eff,tgt_eff3_rec_eff,tgt_eff3_avg_eff,tgt_eff4_rec_eff,tgt_eff4_avg_eff
0,0,1.671730e+18,"@TheFP @VPrasadMDMPH Imagine “both-sidesing” a dude who thinks vaccines cause autism, wifi melts your brain, 5G will control your mind, HIV doesn’t cause AIDS, and the 1918 pandemic was caused by a vaccine not invented yet?\n\nFrom a dude universally derided by his colleagues?\n\nPathetic.",new negative tweets,Yes,Yes,I would rate this tweet a 1. The message appears to focus on criticizing a particular viewpoint rather,Rating: 2\n\nWhile the tweet highlights the importance of addressing misinformation related to HIV and health in,I would rate this tweet a 1. The content of the tweet does not specifically target gay or,Rating: 1 - Not at all\n\nThe tweet does not specifically address issues related to gay or,I would rate this tweet a 2. While it expresses strong emotions about misinformation related to health topics,"I would rate this tweet a 1. The message is overly sarcastic and dismissive, which",Rating: 1 \n\nThis tweet does not provide any clear message or guidance about HIV prevention or testing,Rating: 1 - Not at all\n\nThe tweet does not provide any clear indication or guidance on,"I would rate this tweet a 1 (""Not at all""). The message focuses on criticizing someone's beliefs",I would rate this tweet a 1. The message does not point to any concrete resources for HIV,"Rating: 3 (Agree) \n\nThe tweet conveys a strong stance against misinformation related to health,","I would rate this tweet a 2 (""Neither agree nor disagree""). The tweet expresses frustration with the",1,"I would rate this tweet a 1 (""Disagree""). The message does not appear to be designed",Rating: 1 (Disagree)\n\nThe tweet appears to focus on criticizing misinformation and does not convey,1. Disagree,Rating: 1 (Disagree),1 - Disagree
1,1,1.671730e+18,"@GeorgeTakei And syphilis is good, as well as a woman with pustule vagal warts make for a pleasant bedfellow...",new negative tweets,Yes,Yes,"I would rate this tweet a 1 (""Not at all""). The content is inappropriate and promotes a",Rating: 1 (Not at all)\n\nThe tweet contains inappropriate and insensitive language regarding serious health issues,"I would rate this tweet a 1 (""Not at all""). The message seems to be inappropriate and","I would rate this tweet a 1 - ""Not at all."" The message does not appear to","I would rate this tweet a 1 (""Not at all""). The message is not convincing and appears",Rating: 1 - Not at all\n\nThis tweet contains misleading and inappropriate information about sexually transmitted infections,Rating: 1\n\nThe message in the tweet does not provide any clear or constructive information regarding HIV,Rating: 1 - Not at all\n\nThe tweet does not provide any clear message or indication regarding,Rating: 1 \n\nThis tweet does not indicate any concrete resources for HIV prevention or testing and seems,Rating: 1 (Not at all)\n\nThis tweet does not provide any concrete resources or information related,"I would rate this tweet as 1 - ""Disagree."" The message appears to trivialize sexually",Rating: 1 (Disagree)\n\nThis tweet expresses a negative and misleading perspective on sexually transmitted infections,1,"I would rate this tweet a 1 (""Disagree""). The message seems to trivialize serious health",1,1 - Disagree. \n\nThis tweet seems to trivialize serious health issues and could be viewed as,"I would rate this tweet a 1 (""Disagree""). The message is inappropriate and distasteful","I would rate this tweet a 1 (""Disagree""). The message trivializes serious health issues and"
2,2,1.671730e+18,"@hakii44g @MCU_Direct nah, so far the problem is, that Tenoch removing his condom without her concerns few years ago

In [9]:
neg_data = pd.read_csv('hiv_msgs_working_20241118.csv')  # Replace with your actual file
neg_data = neg_data[neg_data['neg_revisit']=='Yes']
